# Lab 3: Test MCP Server

Tests the 13 tools of the deployed MCP server via the AgentCore Gateway MCP endpoint.

> ⚠️ The first call takes 1-2 minutes to create a Livy session. Subsequent calls respond within seconds.


## Step 1: Setup MCP Client


In [ ]:
import boto3
import json
import time
import os

REGION = (
    boto3.session.Session().region_name
    or os.environ.get("AWS_DEFAULT_REGION")
    or os.environ.get("AWS_REGION")
)
FUNCTION_NAME = "fhir-mcp-server"
lambda_client = boto3.client("lambda", region_name=REGION)

def invoke_tool(tool_name: str, arguments: dict = {}) -> dict:
    """MCP 도구를 Lambda 직접 호출로 테스트합니다."""
    start = time.time()
    resp = lambda_client.invoke(
        FunctionName=FUNCTION_NAME,
        Payload=json.dumps({"toolName": tool_name, "arguments": arguments})
    )
    result = json.loads(resp["Payload"].read())
    elapsed = time.time() - start
    print(f"⏱ {elapsed:.1f}s | {tool_name}")
    if result.get("status") == "error":
        print(f"❌ Error: {result.get('message')}")
    else:
        print(json.dumps(result.get("result", result), indent=2, ensure_ascii=False)[:3000])
    return result

print("Ready!")

Ready!


## Test 1: Schema Discovery

Explores the table structure of the data lake.


In [2]:
# 전체 테이블 목록
invoke_tool("list_tables")

⏱ 1.5s | list_tables
"[{\"table\": \"patient\", \"fqn\": \"`s3tablescatalog`.`data`.`patient`\", \"domain\": \"Administrative\", \"fhir_resource\": \"Patient\", \"description\": \"Patient demographic and administrative information including name, gender, birth date, address, and contact details\", \"column_count\": 24}, {\"table\": \"practitioner\", \"fqn\": \"`s3tablescatalog`.`data`.`practitioner`\", \"domain\": \"Administrative\", \"fhir_resource\": \"Practitioner\", \"description\": \"Healthcare practitioner information including name, qualifications, gender, and contact details\", \"column_count\": 19}, {\"table\": \"organization\", \"fqn\": \"`s3tablescatalog`.`data`.`organization`\", \"domain\": \"Administrative\", \"fhir_resource\": \"Organization\", \"description\": \"Healthcare organization details including name, type, address, and active status\", \"column_count\": 16}, {\"table\": \"location\", \"fqn\": \"`s3tablescatalog`.`data`.`location`\", \"domain\": \"Administrative\

{'status': 'success',
 'result': '[{"table": "patient", "fqn": "`s3tablescatalog`.`data`.`patient`", "domain": "Administrative", "fhir_resource": "Patient", "description": "Patient demographic and administrative information including name, gender, birth date, address, and contact details", "column_count": 24}, {"table": "practitioner", "fqn": "`s3tablescatalog`.`data`.`practitioner`", "domain": "Administrative", "fhir_resource": "Practitioner", "description": "Healthcare practitioner information including name, qualifications, gender, and contact details", "column_count": 19}, {"table": "organization", "fqn": "`s3tablescatalog`.`data`.`organization`", "domain": "Administrative", "fhir_resource": "Organization", "description": "Healthcare organization details including name, type, address, and active status", "column_count": 16}, {"table": "location", "fqn": "`s3tablescatalog`.`data`.`location`", "domain": "Administrative", "fhir_resource": "Location", "description": "Physical location 

In [3]:
# 임상 도메인 테이블만
invoke_tool("list_tables", {"domain": "clinical"})

⏱ 0.0s | list_tables
"[{\"table\": \"encounter\", \"fqn\": \"`s3tablescatalog`.`data`.`encounter`\", \"domain\": \"Clinical\", \"fhir_resource\": \"Encounter\", \"description\": \"Patient encounters (visits) with healthcare providers including type, class, period, reason, and service provider\", \"column_count\": 21}, {\"table\": \"condition\", \"fqn\": \"`s3tablescatalog`.`data`.`condition`\", \"domain\": \"Clinical\", \"fhir_resource\": \"Condition\", \"description\": \"Clinical conditions, diagnoses, and health concerns including onset, clinical status, and verification status\", \"column_count\": 20}, {\"table\": \"observation\", \"fqn\": \"`s3tablescatalog`.`data`.`observation`\", \"domain\": \"Clinical\", \"fhir_resource\": \"Observation\", \"description\": \"Clinical observations and measurements including vital signs, lab results, and assessments with values and units\", \"column_count\": 23}, {\"table\": \"procedure\", \"fqn\": \"`s3tablescatalog`.`data`.`procedure`\", \"domai

{'status': 'success',
 'result': '[{"table": "encounter", "fqn": "`s3tablescatalog`.`data`.`encounter`", "domain": "Clinical", "fhir_resource": "Encounter", "description": "Patient encounters (visits) with healthcare providers including type, class, period, reason, and service provider", "column_count": 21}, {"table": "condition", "fqn": "`s3tablescatalog`.`data`.`condition`", "domain": "Clinical", "fhir_resource": "Condition", "description": "Clinical conditions, diagnoses, and health concerns including onset, clinical status, and verification status", "column_count": 20}, {"table": "observation", "fqn": "`s3tablescatalog`.`data`.`observation`", "domain": "Clinical", "fhir_resource": "Observation", "description": "Clinical observations and measurements including vital signs, lab results, and assessments with values and units", "column_count": 23}, {"table": "procedure", "fqn": "`s3tablescatalog`.`data`.`procedure`", "domain": "Clinical", "fhir_resource": "Procedure", "description": "C

In [ ]:
# patient 스키마 확인
invoke_tool("get_table_schema", {"table_name": "patient"})

⏱ 0.0s | get_table_schema
"{\"table\": \"patient\", \"fqn\": \"`s3tablescatalog`.`data`.`patient`\", \"domain\": \"Administrative\", \"fhir_resource\": \"Patient\", \"description\": \"Patient demographic and administrative information including name, gender, birth date, address, and contact details\", \"columns\": [{\"column_name\": \"addr_cty\", \"expanded_name\": \"address_city\", \"data_type\": \"string\", \"description\": \"City name\", \"semantic_category\": \"address\", \"nullable\": true}, {\"column_name\": \"mlt_bth_ind\", \"expanded_name\": \"multiple_birth_indicator\", \"data_type\": \"boolean\", \"description\": \"Whether patient is part of a multiple birth\", \"semantic_category\": \"demographics\", \"nullable\": true}, {\"column_name\": \"gndr\", \"expanded_name\": \"gender\", \"data_type\": \"string\", \"description\": \"Administrative gender\", \"semantic_category\": \"demographics\", \"nullable\": true}, {\"column_name\": \"tlc_sys\", \"expanded_name\": \"telecom_system

{'status': 'success',
 'result': '{"table": "patient", "fqn": "`s3tablescatalog`.`data`.`patient`", "domain": "Administrative", "fhir_resource": "Patient", "description": "Patient demographic and administrative information including name, gender, birth date, address, and contact details", "columns": [{"column_name": "addr_cty", "expanded_name": "address_city", "data_type": "string", "description": "City name", "semantic_category": "address", "nullable": true}, {"column_name": "mlt_bth_ind", "expanded_name": "multiple_birth_indicator", "data_type": "boolean", "description": "Whether patient is part of a multiple birth", "semantic_category": "demographics", "nullable": true}, {"column_name": "gndr", "expanded_name": "gender", "data_type": "string", "description": "Administrative gender", "semantic_category": "demographics", "nullable": true}, {"column_name": "tlc_sys", "expanded_name": "telecom_system", "data_type": "string", "description": "Telecom system type (phone, email, etc.)", "se

In [2]:
# observation 테이블 관계 확인
invoke_tool("get_table_relationships", {"table_name": "observation"})

⏱ 0.0s | get_table_relationships
"[{\"table\": \"observation\", \"fqn\": \"`s3tablescatalog`.`data`.`observation`\", \"references\": [{\"column\": \"sbj_ref\", \"expanded_name\": \"subject_reference\", \"references_table\": \"patient\"}, {\"column\": \"evt_ref\", \"expanded_name\": \"encounter_reference\", \"references_table\": \"encounter\"}]}]"


{'status': 'success',
 'result': '[{"table": "observation", "fqn": "`s3tablescatalog`.`data`.`observation`", "references": [{"column": "sbj_ref", "expanded_name": "subject_reference", "references_table": "patient"}, {"column": "evt_ref", "expanded_name": "encounter_reference", "references_table": "encounter"}]}]'}

## Test 2: Patient Search

Searches for patients and retrieves detailed information.


In [10]:
# 당뇨 환자 검색
result = invoke_tool("search_patients", {"condition_code": "diabetes", "gender": "female"})

⏱ 13.5s | search_patients
[
  {
    "address_city": "Boston",
    "multiple_birth_indicator": false,
    "gender": "F",
    "telecom_system": "phone",
    "updated_datetime": "2026-03-11T05:52:29.639Z",
    "created_datetime": "2026-03-11T05:52:29.639Z",
    "address_postal_code": "02151",
    "address_country": "US",
    "resource_id": "51de83a3-4b3c-c6d8-f8e7-2171496610e4",
    "identifier_system": "http://hl7.org/fhir/sid/us-ssn",
    "resource_type": "Patient",
    "name_family": "Greenfelder433",
    "telecom_value": "555-727-1460",
    "birth_date": "1979-08-15",
    "identifier_value": "999-90-2324",
    "address_line_1": "841 Quitzon Hollow",
    "address_state": "MA",
    "marital_status": "M",
    "name_given": "Lorri590"
  },
  {
    "address_city": "Amherst",
    "multiple_birth_indicator": false,
    "gender": "F",
    "telecom_system": "phone",
    "updated_datetime": "2026-03-11T05:52:29.639Z",
    "created_datetime": "2026-03-11T05:52:29.639Z",
    "address_postal_code"

In [12]:
# 검색 결과에서 첫 번째 환자 ID로 종합 요약 조회
patients = result.get("result", [])
if patients:
    print(patients[0])
    pid = patients[0]["resource_id"] if isinstance(patients[0], dict) else None
    if pid:
        print(f"\n--- Patient Summary for {pid} ---")
        invoke_tool("get_patient_summary", {"patient_id": pid})

{'address_city': 'Boston', 'multiple_birth_indicator': False, 'gender': 'F', 'telecom_system': 'phone', 'updated_datetime': '2026-03-11T05:52:29.639Z', 'created_datetime': '2026-03-11T05:52:29.639Z', 'address_postal_code': '02151', 'address_country': 'US', 'resource_id': '51de83a3-4b3c-c6d8-f8e7-2171496610e4', 'identifier_system': 'http://hl7.org/fhir/sid/us-ssn', 'resource_type': 'Patient', 'name_family': 'Greenfelder433', 'telecom_value': '555-727-1460', 'birth_date': '1979-08-15', 'identifier_value': '999-90-2324', 'address_line_1': '841 Quitzon Hollow', 'address_state': 'MA', 'marital_status': 'M', 'name_given': 'Lorri590'}

--- Patient Summary for 51de83a3-4b3c-c6d8-f8e7-2171496610e4 ---
⏱ 9.7s | get_patient_summary
{
  "patient": [
    {
      "address_city": "Boston",
      "multiple_birth_indicator": false,
      "gender": "F",
      "telecom_system": "phone",
      "updated_datetime": "2026-03-11T05:52:29.639Z",
      "created_datetime": "2026-03-11T05:52:29.639Z",
      "addr

## Test 3: Clinical Data

Retrieves encounter history, observations, medications, and diagnosis history.

> Uses the `pid` obtained from the test above. If not available, enter it directly in the cell below.


In [13]:
# pid = "직접-입력-환자-ID"  # 필요시 주석 해제

# 진료 이력
invoke_tool("get_encounter_history", {"patient_id": pid})

⏱ 2.0s | get_encounter_history
[
  {
    "reason_code_system": "http://snomed.info/sct",
    "class_code": "AMB",
    "subject_reference": "51de83a3-4b3c-c6d8-f8e7-2171496610e4",
    "period_end_datetime": "2026-02-04T06:14:20.000Z",
    "updated_datetime": "2026-03-11T05:56:01.691Z",
    "type_display": "Encounter for problem (procedure)",
    "created_datetime": "2026-03-11T05:56:01.691Z",
    "resource_id": "51de83a3-4b3c-c6d8-2e11-a3774812b360",
    "identifier_system": "https://github.com/synthetichealth/synthea",
    "organization_reference": "synthea|74ab949d-17ac-3309-83a0-13b4405c66aa",
    "class_system": "http://terminology.hl7.org/CodeSystem/v3-ActCode",
    "resource_type": "Encounter",
    "period_start_datetime": "2026-02-04T05:33:54.000Z",
    "service_provider_reference": "us-npi|9999984591",
    "status": "C",
    "reason_code_value": "60234000",
    "reason_code_display": "Aortic valve regurgitation (disorder)",
    "identifier_value": "51de83a3-4b3c-c6d8-2e11-a37748

{'status': 'success',
 'result': [{'reason_code_system': 'http://snomed.info/sct',
   'class_code': 'AMB',
   'subject_reference': '51de83a3-4b3c-c6d8-f8e7-2171496610e4',
   'period_end_datetime': '2026-02-04T06:14:20.000Z',
   'updated_datetime': '2026-03-11T05:56:01.691Z',
   'type_display': 'Encounter for problem (procedure)',
   'created_datetime': '2026-03-11T05:56:01.691Z',
   'resource_id': '51de83a3-4b3c-c6d8-2e11-a3774812b360',
   'identifier_system': 'https://github.com/synthetichealth/synthea',
   'organization_reference': 'synthea|74ab949d-17ac-3309-83a0-13b4405c66aa',
   'class_system': 'http://terminology.hl7.org/CodeSystem/v3-ActCode',
   'resource_type': 'Encounter',
   'period_start_datetime': '2026-02-04T05:33:54.000Z',
   'service_provider_reference': 'us-npi|9999984591',
   'status': 'C',
   'reason_code_value': '60234000',
   'reason_code_display': 'Aortic valve regurgitation (disorder)',
   'identifier_value': '51de83a3-4b3c-c6d8-2e11-a3774812b360',
   'type_code'

In [14]:
# 관찰 데이터 (혈당)
invoke_tool("get_clinical_observations", {"patient_id": pid, "observation_code": "glucose"})

⏱ 2.0s | get_clinical_observations
[
  {
    "issued_datetime": "2025-09-03T04:29:15.358Z",
    "effective_datetime": "2025-09-03T04:29:15.000Z",
    "subject_reference": "51de83a3-4b3c-c6d8-f8e7-2171496610e4",
    "category_code": "laboratory",
    "updated_datetime": "2026-03-11T06:05:34.286Z",
    "created_datetime": "2026-03-11T06:05:34.286Z",
    "category_display": "Laboratory",
    "value_system": "http://unitsofmeasure.org",
    "resource_id": "51de83a3-4b3c-c6d8-a760-4c045fff174e",
    "resource_type": "Observation",
    "code_text": "Glucose [Mass/volume] in Blood",
    "code_system": "http://loinc.org",
    "status": "F",
    "category_system": "http://terminology.hl7.org/CodeSystem/observation-category",
    "encounter_reference": "51de83a3-4b3c-c6d8-f80d-e8f08666bd63",
    "value_unit": "mg/dL",
    "value_code": "mg/dL",
    "value_quantity": 82.91,
    "code_value": "2339-0",
    "code_display": "Glucose [Mass/volume] in Blood"
  },
  {
    "issued_datetime": "2023-08-30

{'status': 'success',
 'result': [{'issued_datetime': '2025-09-03T04:29:15.358Z',
   'effective_datetime': '2025-09-03T04:29:15.000Z',
   'subject_reference': '51de83a3-4b3c-c6d8-f8e7-2171496610e4',
   'category_code': 'laboratory',
   'updated_datetime': '2026-03-11T06:05:34.286Z',
   'created_datetime': '2026-03-11T06:05:34.286Z',
   'category_display': 'Laboratory',
   'value_system': 'http://unitsofmeasure.org',
   'resource_id': '51de83a3-4b3c-c6d8-a760-4c045fff174e',
   'resource_type': 'Observation',
   'code_text': 'Glucose [Mass/volume] in Blood',
   'code_system': 'http://loinc.org',
   'status': 'F',
   'category_system': 'http://terminology.hl7.org/CodeSystem/observation-category',
   'encounter_reference': '51de83a3-4b3c-c6d8-f80d-e8f08666bd63',
   'value_unit': 'mg/dL',
   'value_code': 'mg/dL',
   'value_quantity': 82.91,
   'code_value': '2339-0',
   'code_display': 'Glucose [Mass/volume] in Blood'},
  {'issued_datetime': '2023-08-30T04:29:15.358Z',
   'effective_dateti

In [ ]:
# 현재 투약 중인 약물
invoke_tool("get_medications", {"patient_id": pid, "active_only": True})

⏱ 2.0s | get_medications
[
  {
    "updated_datetime": "2026-03-11T06:00:06.686Z",
    "medication_code_system": "http://www.nlm.nih.gov/research/umls/rxnorm",
    "category_display": "Community",
    "resource_id": "51de83a3-4b3c-c6d8-4ee6-8be4f97765e3",
    "intent": "O",
    "medication_code_value": "1367439",
    "subject_reference": "51de83a3-4b3c-c6d8-f8e7-2171496610e4",
    "category_code": "community",
    "created_datetime": "2026-03-11T06:00:06.686Z",
    "authored_datetime": "2024-09-19T11:49:54.000Z",
    "resource_type": "MedicationRequest",
    "status": "C",
    "category_system": "http://terminology.hl7.org/CodeSystem/medicationrequest-category",
    "encounter_reference": "51de83a3-4b3c-c6d8-1020-859a341f14be",
    "request_reference": "us-npi|9999992396",
    "medication_code_display": "21 DAY ethinyl estradiol 0.000625 MG/HR / etonogestrel 0.005 MG/HR Vaginal System [NuvaRing]"
  },
  {
    "updated_datetime": "2026-03-11T06:00:06.686Z",
    "category_display": "Outp

{'status': 'success',
 'result': [{'updated_datetime': '2026-03-11T06:00:06.686Z',
   'medication_code_system': 'http://www.nlm.nih.gov/research/umls/rxnorm',
   'category_display': 'Community',
   'resource_id': '51de83a3-4b3c-c6d8-4ee6-8be4f97765e3',
   'intent': 'O',
   'medication_code_value': '1367439',
   'subject_reference': '51de83a3-4b3c-c6d8-f8e7-2171496610e4',
   'category_code': 'community',
   'created_datetime': '2026-03-11T06:00:06.686Z',
   'authored_datetime': '2024-09-19T11:49:54.000Z',
   'resource_type': 'MedicationRequest',
   'status': 'C',
   'category_system': 'http://terminology.hl7.org/CodeSystem/medicationrequest-category',
   'encounter_reference': '51de83a3-4b3c-c6d8-1020-859a341f14be',
   'request_reference': 'us-npi|9999992396',
   'medication_code_display': '21 DAY ethinyl estradiol 0.000625 MG/HR / etonogestrel 0.005 MG/HR Vaginal System [NuvaRing]'},
  {'updated_datetime': '2026-03-11T06:00:06.686Z',
   'category_display': 'Outpatient',
   'resource_id

In [17]:
# 진단 이력
invoke_tool("get_diagnosis_history", {"patient_id": pid})

⏱ 2.0s | get_diagnosis_history
[
  {
    "clinical_status_system": "http://terminology.hl7.org/CodeSystem/condition-clinical",
    "abatement_datetime": "2025-11-14T09:29:15.000Z",
    "clinical_status_code": "RS",
    "subject_reference": "51de83a3-4b3c-c6d8-f8e7-2171496610e4",
    "category_code": "encounter-diagnosis",
    "updated_datetime": "2026-03-11T05:58:12.079Z",
    "created_datetime": "2026-03-11T05:58:12.079Z",
    "category_display": "Encounter Diagnosis",
    "recorded_date": "2025-11-02",
    "resource_id": "51de83a3-4b3c-c6d8-94eb-30b51e7d2779",
    "resource_type": "Condition",
    "code_text": "Acute viral pharyngitis (disorder)",
    "verification_status_code": "C",
    "code_system": "http://snomed.info/sct",
    "onset_datetime": "2025-11-02T19:29:15.000Z",
    "category_system": "http://terminology.hl7.org/CodeSystem/condition-category",
    "encounter_reference": "51de83a3-4b3c-c6d8-27f0-779b5efd2f5d",
    "verification_status_system": "http://terminology.hl7.or

{'status': 'success',
 'result': [{'clinical_status_system': 'http://terminology.hl7.org/CodeSystem/condition-clinical',
   'abatement_datetime': '2025-11-14T09:29:15.000Z',
   'clinical_status_code': 'RS',
   'subject_reference': '51de83a3-4b3c-c6d8-f8e7-2171496610e4',
   'category_code': 'encounter-diagnosis',
   'updated_datetime': '2026-03-11T05:58:12.079Z',
   'created_datetime': '2026-03-11T05:58:12.079Z',
   'category_display': 'Encounter Diagnosis',
   'recorded_date': '2025-11-02',
   'resource_id': '51de83a3-4b3c-c6d8-94eb-30b51e7d2779',
   'resource_type': 'Condition',
   'code_text': 'Acute viral pharyngitis (disorder)',
   'verification_status_code': 'C',
   'code_system': 'http://snomed.info/sct',
   'onset_datetime': '2025-11-02T19:29:15.000Z',
   'category_system': 'http://terminology.hl7.org/CodeSystem/condition-category',
   'encounter_reference': '51de83a3-4b3c-c6d8-27f0-779b5efd2f5d',
   'verification_status_system': 'http://terminology.hl7.org/CodeSystem/condition-

## Test 4: Financial & Analytics


In [18]:
# 청구 요약
invoke_tool("get_claim_summary", {"patient_id": pid})

⏱ 2.0s | get_claim_summary
[
  {
    "subject_reference": "51de83a3-4b3c-c6d8-f8e7-2171496610e4",
    "provider_reference": "synthea|0f2c58ec-5323-3c0f-9bdf-af0df7eb2a2b",
    "created_datetime": "1993-09-17T05:29:15.000Z",
    "resource_id": "51de83a3-4b3c-c6d8-dbcc-0e86fbe83d2f",
    "total_amount": 146.18,
    "resource_type": "Claim",
    "status": "A",
    "use_code": "CL",
    "priority": "normal",
    "total_currency": "USD",
    "system_created_datetime": "2026-03-11T05:56:54.291Z",
    "type_code": "institutional",
    "type_system": "http://terminology.hl7.org/CodeSystem/claim-type",
    "system_updated_datetime": "2026-03-11T05:56:54.291Z"
  },
  {
    "subject_reference": "51de83a3-4b3c-c6d8-f8e7-2171496610e4",
    "provider_reference": "synthea|0f2c58ec-5323-3c0f-9bdf-af0df7eb2a2b",
    "created_datetime": "1993-10-24T05:29:15.000Z",
    "resource_id": "51de83a3-4b3c-c6d8-5e98-c82c102ddaf0",
    "total_amount": 87.71,
    "resource_type": "Claim",
    "status": "A",
    "u

{'status': 'success',
 'result': [{'subject_reference': '51de83a3-4b3c-c6d8-f8e7-2171496610e4',
   'provider_reference': 'synthea|0f2c58ec-5323-3c0f-9bdf-af0df7eb2a2b',
   'created_datetime': '1993-09-17T05:29:15.000Z',
   'resource_id': '51de83a3-4b3c-c6d8-dbcc-0e86fbe83d2f',
   'total_amount': 146.18,
   'resource_type': 'Claim',
   'status': 'A',
   'use_code': 'CL',
   'priority': 'normal',
   'total_currency': 'USD',
   'system_created_datetime': '2026-03-11T05:56:54.291Z',
   'type_code': 'institutional',
   'type_system': 'http://terminology.hl7.org/CodeSystem/claim-type',
   'system_updated_datetime': '2026-03-11T05:56:54.291Z'},
  {'subject_reference': '51de83a3-4b3c-c6d8-f8e7-2171496610e4',
   'provider_reference': 'synthea|0f2c58ec-5323-3c0f-9bdf-af0df7eb2a2b',
   'created_datetime': '1993-10-24T05:29:15.000Z',
   'resource_id': '51de83a3-4b3c-c6d8-5e98-c82c102ddaf0',
   'total_amount': 87.71,
   'resource_type': 'Claim',
   'status': 'A',
   'use_code': 'CL',
   'priority':

In [19]:
# 케어 갭 분석
invoke_tool("detect_care_gaps", {"patient_id": pid})

⏱ 5.8s | detect_care_gaps
{
  "immunizations": [
    {
      "vaccine_code_display": "Influenza, split virus, trivalent, PF",
      "primary_source": true,
      "updated_datetime": "2026-03-11T06:02:34.170Z",
      "resource_id": "51de83a3-4b3c-c6d8-b9fb-3af1c2707540",
      "occurrence_datetime": "2016-11-02T04:29:15.000Z",
      "vaccine_code_system": "http://hl7.org/fhir/sid/cvx",
      "vaccine_code_value": "140",
      "subject_reference": "51de83a3-4b3c-c6d8-f8e7-2171496610e4",
      "created_datetime": "2026-03-11T06:02:34.170Z",
      "resource_type": "Immunization",
      "status": "C",
      "encounter_reference": "51de83a3-4b3c-c6d8-6574-21b962293d57",
      "location_reference": "synthea|c268163c-1a0b-3f49-bb14-eccd493ce6e5"
    },
    {
      "vaccine_code_display": "Influenza, split virus, trivalent, PF",
      "primary_source": true,
      "updated_datetime": "2026-03-11T06:02:34.170Z",
      "resource_id": "51de83a3-4b3c-c6d8-15f5-d6c5907c3fde",
      "occurrence_datet

{'status': 'success',
 'result': {'immunizations': [{'vaccine_code_display': 'Influenza, split virus, trivalent, PF',
    'primary_source': True,
    'updated_datetime': '2026-03-11T06:02:34.170Z',
    'resource_id': '51de83a3-4b3c-c6d8-b9fb-3af1c2707540',
    'occurrence_datetime': '2016-11-02T04:29:15.000Z',
    'vaccine_code_system': 'http://hl7.org/fhir/sid/cvx',
    'vaccine_code_value': '140',
    'subject_reference': '51de83a3-4b3c-c6d8-f8e7-2171496610e4',
    'created_datetime': '2026-03-11T06:02:34.170Z',
    'resource_type': 'Immunization',
    'status': 'C',
    'encounter_reference': '51de83a3-4b3c-c6d8-6574-21b962293d57',
    'location_reference': 'synthea|c268163c-1a0b-3f49-bb14-eccd493ce6e5'},
   {'vaccine_code_display': 'Influenza, split virus, trivalent, PF',
    'primary_source': True,
    'updated_datetime': '2026-03-11T06:02:34.170Z',
    'resource_id': '51de83a3-4b3c-c6d8-15f5-d6c5907c3fde',
    'occurrence_datetime': '2018-08-08T04:29:15.000Z',
    'vaccine_code_s

In [22]:
# 인구 건강 지표 - 당뇨 60대
invoke_tool("get_population_health_metrics", {"condition_code": "diabetes", "age_group": "60-69"})

⏱ 6.0s | get_population_health_metrics
[
  {
    "age_group": "60-69",
    "gender": "F",
    "condition_display": "Prediabetes (finding)",
    "patient_count": 47
  },
  {
    "age_group": "60-69",
    "gender": "M",
    "condition_display": "Prediabetes (finding)",
    "patient_count": 43
  },
  {
    "age_group": "60-69",
    "gender": "F",
    "condition_display": "Diabetes mellitus type 2 (disorder)",
    "patient_count": 12
  },
  {
    "age_group": "60-69",
    "gender": "M",
    "condition_display": "Disorder of kidney due to diabetes mellitus (disorder)",
    "patient_count": 10
  },
  {
    "age_group": "60-69",
    "gender": "M",
    "condition_display": "Diabetes mellitus type 2 (disorder)",
    "patient_count": 8
  },
  {
    "age_group": "60-69",
    "gender": "F",
    "condition_display": "Microalbuminuria due to type 2 diabetes mellitus (disorder)",
    "patient_count": 8
  },
  {
    "age_group": "60-69",
    "gender": "F",
    "condition_display": "Disorder of kidney 

{'status': 'success',
 'result': [{'age_group': '60-69',
   'gender': 'F',
   'condition_display': 'Prediabetes (finding)',
   'patient_count': 47},
  {'age_group': '60-69',
   'gender': 'M',
   'condition_display': 'Prediabetes (finding)',
   'patient_count': 43},
  {'age_group': '60-69',
   'gender': 'F',
   'condition_display': 'Diabetes mellitus type 2 (disorder)',
   'patient_count': 12},
  {'age_group': '60-69',
   'gender': 'M',
   'condition_display': 'Disorder of kidney due to diabetes mellitus (disorder)',
   'patient_count': 10},
  {'age_group': '60-69',
   'gender': 'M',
   'condition_display': 'Diabetes mellitus type 2 (disorder)',
   'patient_count': 8},
  {'age_group': '60-69',
   'gender': 'F',
   'condition_display': 'Microalbuminuria due to type 2 diabetes mellitus (disorder)',
   'patient_count': 8},
  {'age_group': '60-69',
   'gender': 'F',
   'condition_display': 'Disorder of kidney due to diabetes mellitus (disorder)',
   'patient_count': 8},
  {'age_group': '60-

## Test 5: Custom Query (Text-to-SQL)

Executes free-form SQL queries.


In [28]:
# 가장 많이 처방된 약물 Top 10
invoke_tool("run_custom_query", {
    "query": """
    SELECT medication_code_display, COUNT(*) as cnt
    FROM s3tablescatalog.data.medication_request
    GROUP BY medication_code_display
    ORDER BY cnt DESC
    LIMIT 10
    """
})

⏱ 2.0s | run_custom_query
[
  {
    "cnt": 25239
  },
  {
    "medication_code_display": "lisinopril 10 MG Oral Tablet",
    "cnt": 6802
  },
  {
    "medication_code_display": "insulin isophane, human 70 UNT/ML / insulin, regular, human 30 UNT/ML Injectable Suspension [Humulin]",
    "cnt": 6786
  },
  {
    "medication_code_display": "Hydrochlorothiazide 25 MG Oral Tablet",
    "cnt": 5662
  },
  {
    "medication_code_display": "amLODIPine 2.5 MG Oral Tablet",
    "cnt": 4882
  },
  {
    "medication_code_display": "24 HR Metformin hydrochloride 500 MG Extended Release Oral Tablet",
    "cnt": 3243
  },
  {
    "medication_code_display": "Alendronic acid 10 MG Oral Tablet",
    "cnt": 2198
  },
  {
    "medication_code_display": "24 HR tacrolimus 1 MG Extended Release Oral Tablet [Envarsus]",
    "cnt": 1779
  },
  {
    "medication_code_display": "Simvastatin 10 MG Oral Tablet",
    "cnt": 1253
  },
  {
    "medication_code_display": "12 HR Hydrocodone Bitartrate 10 MG Extended Rel

{'status': 'success',
 'result': [{'cnt': 25239},
  {'medication_code_display': 'lisinopril 10 MG Oral Tablet', 'cnt': 6802},
  {'medication_code_display': 'insulin isophane, human 70 UNT/ML / insulin, regular, human 30 UNT/ML Injectable Suspension [Humulin]',
   'cnt': 6786},
  {'medication_code_display': 'Hydrochlorothiazide 25 MG Oral Tablet',
   'cnt': 5662},
  {'medication_code_display': 'amLODIPine 2.5 MG Oral Tablet', 'cnt': 4882},
  {'medication_code_display': '24 HR Metformin hydrochloride 500 MG Extended Release Oral Tablet',
   'cnt': 3243},
  {'medication_code_display': 'Alendronic acid 10 MG Oral Tablet',
   'cnt': 2198},
  {'medication_code_display': '24 HR tacrolimus 1 MG Extended Release Oral Tablet [Envarsus]',
   'cnt': 1779},
  {'medication_code_display': 'Simvastatin 10 MG Oral Tablet', 'cnt': 1253},
  {'medication_code_display': '12 HR Hydrocodone Bitartrate 10 MG Extended Release Oral Capsule',
   'cnt': 981}]}

In [ ]:
# DML 차단 테스트 (에러가 나야 정상)
invoke_tool("run_custom_query", {"query": "DROP TABLE patient_registry"})

## ✅ Test Summary

| Category | Tools | Status |
|----------|-------|--------|
| Schema Discovery | list_tables, get_table_schema, get_table_relationships | |
| Patient | search_patients, get_patient_summary | |
| Clinical | get_encounter_history, get_clinical_observations, get_medications, get_diagnosis_history | |
| Financial | get_claim_summary | |
| Analytics | detect_care_gaps, get_population_health_metrics | |
| Query | run_custom_query (SELECT ✅, DML ❌) | |
